# Rede neural convolucional com transformada Wavelet no pipeline 


## 1. Dataset - MiniMIAS

### 1.1 Rodar localmente o programa

- Para isso é necessário baixar e extrair o dataset para a máquina local. 

**Link:** https://www.repository.cam.ac.uk/items/b6a97f0c-3b9b-40ad-8f18-3d121eef1459

- Baixe o dataset em Files;
- Após extrair em uma pasta os arquivos. Comando via terminal: _unzip MIAS-DB.zip_

In [5]:
import os

pasta = "dataset-mias/"

arquivos = os.listdir(pasta)
contador = 0

for arquivo in arquivos:
    if ".pgm" in arquivo:
        contador += 1

print(contador)

322


### 1.2 Classificando as imagnes

In [28]:
import os
import random
import shutil
from PyPDF2 import PdfReader

pasta_imagens = "dataset-mias/"

com_nodulo = []
sem_nodulo = []

# LER PDF
reader = PdfReader("dataset-mias/00README.pdf")
linhas = []

for pagina in reader.pages:
    texto = pagina.extract_text()
    linhas.extend(texto.split("\n"))

# PROCESSAR
for linha in linhas:
    if "mdb" in linha:
        partes = linha.split()
        nome = partes[0] + ".pgm"
        
        if "NORM" in linha:
            sem_nodulo.append(nome)
        else:
            com_nodulo.append(nome)

# DIVIDIR
def dividir(lista, pasta_train, pasta_test):
    random.shuffle(lista)
    corte = int(0.7 * len(lista))
    
    for nome in lista[:corte]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_train, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)
    
    for nome in lista[corte:]:
        origem = os.path.join(pasta_imagens, nome)
        destino = os.path.join(pasta_test, nome)
        if os.path.exists(origem):
            shutil.copy(origem, destino)

# PASTAS
os.makedirs("dataset/train/com_nodulo", exist_ok=True)
os.makedirs("dataset/train/sem_nodulo", exist_ok=True)
os.makedirs("dataset/test/com_nodulo", exist_ok=True)
os.makedirs("dataset/test/sem_nodulo", exist_ok=True)

# EXECUTAR
dividir(com_nodulo, "dataset/train/com_nodulo", "dataset/test/com_nodulo")
dividir(sem_nodulo, "dataset/train/sem_nodulo", "dataset/test/sem_nodulo")

print("Finalizado!")

Finalizado!


### 1.3 Carregando dataset para CNN

#### Data Augmentation (aumento de dados)

Agora para o modelo 4 vamos aumentar nosso dataset simplesmente efetuando rotações nas imagnes mamográficas.

Dessa forma, vamos aplicar Data Augmentation nos dados de treino e manter os dados de teste apenas redimensionado com a imagem limpa.

Além disso, foi ajustado a resolução da imagem para 128 x 128 e aplicado uma técnica chamada CLAHE, que aumenta o contraste da imagem.

In [29]:
import cv2
import numpy as np
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Defina a classe do CLAHE primeiro 
class ApplyCLAHE:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        # Converte PIL para Numpy para usar OpenCV
        img_np = np.array(img)
        # Aplica o realce de contraste
        img_clahe = self.clahe.apply(img_np)
        # Volta para PIL para o resto do pipeline
        return Image.fromarray(img_clahe)

# 2. Treino
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    ApplyCLAHE(),  # realce na imagem
    transforms.Resize((128, 128)),
    # --- Data Augmentation ---
    transforms.RandomHorizontalFlip(p=0.5), 
    transforms.RandomRotation(15),          
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))    
])

# 3. Teste
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    ApplyCLAHE(),              
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder(root="dataset/train", transform=train_transform)
test_dataset = datasets.ImageFolder(root="dataset/test", transform=test_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

## 2. Criando uma camada Wavelet usando PyTorch

#### WaveletLayer Treinável

Agora vamos permiti que a rede aprenda e melhore os pesos para a Wavelet automaticamente (self.weight = nn.Parameter(kernel)).

In [30]:
# Camada Wavelet 
import torch
import torch.nn as nn
import torch.nn.functional as F

class WaveletLayer(nn.Module):
    # Passamos os canais de entrada
    def __init__(self, in_channels):  
        super(WaveletLayer, self).__init__()

        # Filtro Haar LL
        kernel_base = torch.tensor([[0.5, 0.5],
                               [0.5, 0.5]], dtype=torch.float32)
        
        # Formato (out, in, h, w) - para wavelet, out = in
        kernel = kernel_base.repeat(in_channels, 1, 1, 1)
        
        # Transformamos em nn.Parameter para ser TREINÁVEL
        self.weight = nn.Parameter(kernel)

    def forward(self, x):
        # para ter um kernel 2 x 2
        x_padded = F.pad(x, (0, 1, 0, 1), mode='reflect')
        # Como o peso já tem o tamanho correto, basta aplicar a convolução
        # O parâmetro chamado "groups" garante que cada canal seja processado isoladamente
        # stride=1 - para usar a wavelet apenas como realçe e não como redução da imagem
        return F.conv2d(x_padded, self.weight, stride=1, groups=x.shape[1])


## 3. Modelo com 3 convoluções de CNN integrando a camada Wavelet e com Data Augmentation

<img src="img/waveletCNN-modelo6.png" width="700" title="Dica de ferramenta">

In [31]:
"""
MODELO COM A WAVELET PARA REALCE E 2 CONVOLUÇÕES ("Wavelet Realce -> Conv -> Conv")
"""
class WaveletRealceCNN(nn.Module):
    def __init__(self):
        super(WaveletRealceCNN, self).__init__()

        # 1. Wavelet de Realce (Mantém 128x128)
        self.wavelet_realce = WaveletLayer(in_channels=1)

        # 2. Sequência Conv-Conv (Extração Profunda)
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.LeakyReLU(), # SUBSTITUINDO O RELU
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2) # resolução: 128 -> 64
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2) # resolução: 64 -> 32
        )

        # Camada Final
        self.gap = nn.AdaptiveAvgPool2d(1) # Transforma tudo em um vetor de 64
        self.fc = nn.Linear(64, 2)


    def forward(self, x):
        # Primeiro realça com Wavelet treinável
        x = self.wavelet_realce(x)
        
        # Depois passa pelos blocos de visão
        x = self.block1(x)
        x = self.block2(x)
        
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

### 3.1 Treinando o modelo

#### Treinamento otimizado

Calcula a média de erro de todos os lotes e mostra a porcentagem de acertos enquanto o modelo aprende.

Aalém disso código já inclui o .to(device), o que acelera o treino em até 10x caso seja rodado em ambiente que tenha uma placa de vídeo ou estiver usando o Google Colab.

In [33]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1. Instanciar o modelo novo
model = WaveletRealceCNN()

# 2. Definir o dispositivo (GPU ou CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 3. Definir a função de perda
criterion = nn.CrossEntropyLoss()

# 4. Definir o otimizador (Passando os parâmetros do NOVO modelo)
# o weight_decay para ajudar a chegar nos 90%
# optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# testando outro tipo de otimizador
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

# O Scheduler vai reduzir o aprendizado quando a rede parar de evoluir
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

epochs = 50

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    corretos_treino = 0
    total_treino = 0
    
    for imagens, labels in train_loader:
        # Enviar dados para o dispositivo (CPU ou GPU)
        imagens, labels = imagens.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(imagens)
        loss = criterion(outputs, labels)

        # Backward pass e otimização
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Estatísticas de Treino
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total_treino += labels.size(0)
        corretos_treino += (predicted == labels).sum().item()

    # Cálculos das médias da época
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * corretos_treino / total_treino
    
    print(f"Época [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Acc Treino: {epoch_acc:.2f}%")

    scheduler.step(epoch_acc)

    # Valida no test_loader a cada 5 épocas para ver se o Overfitting (Sobreajuste) parou
    if (epoch + 1) % 5 == 0:
        model.eval()
        corretos_teste = 0
        total_teste = 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                _, pred = torch.max(out, 1)
                total_teste += lbls.size(0)
                corretos_teste += (pred == lbls).sum().item()
        print(f">>> Acurácia de Teste: {100 * corretos_teste / total_teste:.2f}%")

Época [1/50] - Loss: 0.6617 - Acc Treino: 63.84%
Época [2/50] - Loss: 0.6539 - Acc Treino: 63.84%
Época [3/50] - Loss: 0.6540 - Acc Treino: 62.05%
Época [4/50] - Loss: 0.6491 - Acc Treino: 63.84%
Época [5/50] - Loss: 0.6461 - Acc Treino: 63.39%
>>> Acurácia de Teste: 62.50%
Época [6/50] - Loss: 0.6532 - Acc Treino: 62.05%
Época [7/50] - Loss: 0.6622 - Acc Treino: 63.84%
Época [8/50] - Loss: 0.6478 - Acc Treino: 63.84%
Época [9/50] - Loss: 0.6448 - Acc Treino: 63.84%
Época [10/50] - Loss: 0.6453 - Acc Treino: 62.95%
>>> Acurácia de Teste: 63.54%
Época [11/50] - Loss: 0.6482 - Acc Treino: 63.39%
Época [12/50] - Loss: 0.6406 - Acc Treino: 63.39%
Época [13/50] - Loss: 0.6467 - Acc Treino: 64.29%
Época [14/50] - Loss: 0.6463 - Acc Treino: 63.39%
Época [15/50] - Loss: 0.6435 - Acc Treino: 63.84%
>>> Acurácia de Teste: 62.50%
Época [16/50] - Loss: 0.6438 - Acc Treino: 63.84%
Época [17/50] - Loss: 0.6528 - Acc Treino: 63.84%
Época [18/50] - Loss: 0.6446 - Acc Treino: 62.95%
Época [19/50] - Los

### 3.2 Testando o modelo

In [34]:
model.eval()
corretos = 0
total = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

with torch.no_grad():
    for imagens, labels in test_loader:
        
        imagens, labels = imagens.to(device), labels.to(device)

        outputs = model(imagens)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        corretos += (predicted == labels).sum().item()

print(f"Acurácia: {100 * corretos / total:.2f}%")

Acurácia: 62.50%
